<a href="https://colab.research.google.com/github/Jenn2626/Python/blob/main/e_commerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PRACTICE 5:
## Description
> Use the e-commerce dataset and perform the following tasks:

1. Calculate Average Rating and Number of Reviews by Product Categories
Hint: Group the data by product categories. Use aggregation functions like mean() for the average rating and count() for the number of reviews.
2. Number of Orders, and Sales by Product Categories
Hint: Group by product categories and calculate the total number of orders, and total sales (payment_value).
3. Which States Have the Highest Number of Customers?
Hint: Group the data by the state and count the number of distinct customers. You can use groupby() and nunique() to count unique customers by state.
4. Top 5 Customers Generating the Most Revenue
Hint: Group by customer and sum the total revenue (payment_value). Sort the results in descending order and select the top 5.
5. umber of Orders for Each Category (“Morning”, “Afternoon”, “Evening”, “Night”)
Categorize purchase time based on the order_purchase_timestamp hour:
+ “Morning” for hours between 5 AM and 12 PM.
+ “Afternoon” for hours between 12 PM and 5 PM.
 “Evening” for hours between 5 PM and 9 PM.
+ “Night” for hours between 9 PM and 5 AM.
Hint: Categorize the purchase time based on the order_purchase_timestamp hour. Use the .dt accessor to extract the hour, then categorize the hour into “Morning”, “Afternoon”, “Evening”, or “Night”.
6. Total Number of Orders, and Sales by Day of the Week
Hint: Extract the day of the week from the order_purchase_timestamp and group by this day. You can use .dt.day_name() or .dt.weekday to get the day name.
7. Average Rating Based on the Difference Between the Estimated and Actual Delivery Dates
Hint:
+ Calculate the difference between the order_estimated_delivery_date and the order_delivered_customer_date in terms of days.
+ Categorize the delivery status based on the following conditions:
+ If the difference is more than 10 days, categorize it as “late over 10 days”.
+ If the difference is between 5 and 10 days, categorize it as “late from 5 days to 10 days”.
+ If the difference is less than 5 days, categorize it as “late under 5 days”.
+ If the difference is 0 or greater, categorize it as “on time delivery”.
+ Calculate the average rating based on the est_to_deliver_detail column.
Notes:

### Make sure to check the datatype of the columns you are working with, especially dates. You may need to use pd.to_datetime() for conversion.
### For categorizing timestamps or dates, use .dt to extract the components such as day, hour, or month.
### When performing aggregations, use groupby() to group by the relevant column and apply aggregation functions such as mean(), sum(), count(), or nunique().

In [16]:
import pandas as pd
df_order_items = pd.read_csv('/content/order_items.csv', on_bad_lines='skip')
df_order_payments = pd.read_csv('/content/order_payments.csv')
df_order_reviews = pd.read_csv('/content/order_reviews.csv')
df_products = pd.read_csv('/content/products.csv')
df_sellers = pd.read_csv('/content/sellers.csv')
df_product_category_name_translation = pd.read_csv('/content/product_category_name_translation.csv')
df_geolocation = pd.read_csv('/content/geolocation.csv')
df_customers = pd.read_csv('/content/customers.csv')

In [8]:
import pandas as pd
df_orders = pd.read_csv('/content/orders.csv')

In [ ]:
dfs = [df_product_category_name_translation, df_customers, df_geolocation, df_order_items, df_orders, df_order_payments,
       df_products, df_order_reviews, df_sellers]
df_names = ['df_product_category_name_translation', 'df_customers', 'df_geolocation', 'df_order_items', 'df_orders',
            'df_order_payments', 'df_products', 'df_order_reviews', 'df_sellers']

summary_data = []

# Dùng phương thức apply để lấy thông tin cho từng bảng
for df, name in zip(dfs, df_names):
    shape = df.shape  # (số dòng, số cột)
    columns = list(df.columns)  # Tên các cột
    summary_data.append({
        'Tên bảng': name,
        'Shape': shape,
        'Tên cột': columns,
    })

# Tạo DataFrame từ thông tin tổng hợp
summary_df = pd.DataFrame(summary_data)
summary_df


,Tên bảng,Shape,Tên cột
0,df_product_category_name_translation,"(71, 2)","[product_category_name, product_category_name_..."
1,df_customers,"(99441, 5)","[customer_id, customer_unique_id, customer_zip..."
2,df_geolocation,"(1000163, 5)","[geolocation_zip_code_prefix, geolocation_lat,..."
3,df_order_items,"(112650, 7)","[order_id, order_item_id, product_id, seller_i..."
4,df_orders,"(99441, 8)","[order_id, customer_id, order_status, order_pu..."
5,df_order_payments,"(103886, 5)","[order_id, payment_sequential, payment_type, p..."
6,df_products,"(32951, 9)","[product_id, product_category_name, product_na..."
7,df_order_reviews,"(99224, 7)","[review_id, order_id, review_score, review_com..."
8,df_sellers,"(3095, 4)","[seller_id, seller_zip_code_prefix, seller_cit..."


In [ ]:
# Calculate Average Rating and Number of Reviews by Product Categories
import pandas as pd
merge_order_reviews = pd.merge(df_order_items,df_order_reviews, how='inner', on = 'order_id')
merge_order_reviews_product = pd.merge(merge_order_reviews,df_products, how = 'inner', on = 'product_id')
cal_avg_count_review = merge_order_reviews_product.groupby('product_category_name').agg(
    avg_rating = ('review_score','mean'),
    number_view = ('review_id','count')).reset_index()
cal_avg_count_review = cal_avg_count_review.round(2)
cal_avg_count_review = cal_avg_count_review.sort_values(by = 'avg_rating', ascending = False)
cal_avg_count_review



,product_category_name,avg_rating,number_view
17,cds_dvds_musicais,4.64,14
37,fashion_roupa_infanto_juvenil,4.50,8
48,livros_interesse_geral,4.45,549
22,construcao_ferramentas_ferramentas,4.44,99
41,flores,4.42,31
...,...,...,...
55,moveis_escritorio,3.49,1687
60,pc_gamer,3.33,9
65,portateis_cozinha_e_preparadores_de_alimentos,3.27,15
42,fraldas_higiene,3.26,39


In [ ]:
# Number of Orders, and Sales by Product Categories
import pandas as pd
merge_items_payment = pd.merge(df_order_items,df_order_payments,how = 'inner',on = 'order_id')
merge_items_payment_product = pd.merge(merge_items_payment,df_products, how = 'inner', on = 'product_id')
result_order_sale = merge_items_payment_product.groupby('product_category_name').agg(
    total_order = ('order_id','count'),
    total_sales = ('payment_value','sum')).reset_index()

result_order_sale = result_order_sale.sort_values('total_sales',ascending=False)
result_order_sale
#

,product_category_name,total_order,total_sales
13,cama_mesa_banho,11823,1712553.67
11,beleza_saude,9972,1657373.12
44,informatica_acessorios,8082,1585330.45
54,moveis_decoracao,8744,1430176.39
66,relogios_presentes,6201,1429216.68
...,...,...,...
60,pc_gamer,10,2174.43
15,casa_conforto_2,31,1710.54
17,cds_dvds_musicais,14,1199.43
37,fashion_roupa_infanto_juvenil,8,785.67


In [ ]:
# Which States Have the Highest Number of Unique Customers(use:nunique)?
import pandas as pd
customer_state = df_customers.groupby('customer_state').agg(
    total_customer = ('customer_id','nunique')).reset_index()
customer_state = customer_state.sort_values('total_customer', ascending=False)
customer_state

,customer_state,total_customer
25,SP,41746
18,RJ,12852
10,MG,11635
22,RS,5466
17,PR,5045
23,SC,3637
4,BA,3380
6,DF,2140
7,ES,2033
8,GO,2020


In [ ]:
# Top 5 Customers Generating the Most Revenue
import pandas as pd
merge_order_payment = pd.merge(df_orders,df_order_payments,how = 'inner',on = 'order_id')
result_customer_revenue = merge_order_payment.groupby('customer_id').agg(
    total_revenue = ('payment_value','sum')).reset_index()

result_customer_revenue = result_customer_revenue.sort_values('total_revenue', ascending=False)
result_customer_revenue = result_customer_revenue.head(5)
result_customer_revenue

,customer_id,total_revenue
8546,1617b1357756262bfa56ab541c47bc16,13664.08
91984,ec5b2ba62e574342386871631fafd3fc,7274.88
77521,c6e2731c5b391845f6800c97401a43a9,6929.31
95123,f48d464a0baaea338cb25f816991ab1f,6922.21
24771,3fd6777bbce08a352fddd04e4a7cc8f6,6726.66


In [ ]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [ ]:
# Number of Orders for Each Category
# (“Morning” for hours between 5 AM and 12 PM, “Afternoon” for hours between 12 PM and 5 PM, “Evening”for hours between 5 PM and 9 PM, “Night”for hours between 9 PM and 5 AM)
# Categorize purchase time based on the order_purchase_timestamp hour:
import pandas as pd
# Vi order_purchase_timestamp type string nen phai chuyen thanh datetime
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
# Extract order_purchase_timestamp thanh hours
df_orders['order_purchase_timestamp'] = df_orders['order_purchase_timestamp'].dt.hour
df_orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,10,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,20,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,8,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,19,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,21,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,9,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,12,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,14,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,21,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [ ]:
# Category Purchase Hours
def order_time_category (time):
  if 5 < time < 12:
    return 'Morning'
  elif 12 <= time < 5:
    return 'Afternoon'
  elif 5 <= time < 9:
    return 'Evening'
  else:
    return 'Night'

df_orders['time_category'] = df_orders['order_purchase_timestamp'].apply(order_time_category)

# Calculate order each Category Purchase Hours
cal_order_time = df_orders.groupby('time_category').agg(
    total_order = ('order_id','count')).reset_index
cal_order_time()


,time_category,total_order
0,Evening,188
1,Morning,22240
2,Night,77013


In [ ]:
# Total Number of Orders, and Sales by Day of the Week

import pandas as pd
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_purchase_timestamp'] = df_orders['order_purchase_timestamp'].dt.dayofweek
df_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,time_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,3,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,Morning
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,3,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,Night
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,3,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,Morning
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,3,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,Night
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,3,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,Night
...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,3,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00,Morning
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,3,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,Night
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,3,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,Night
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,3,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,Night


In [ ]:
merge_order_payment_week = pd.merge(df_orders,df_order_payments, how = 'inner', on = 'order_id')
result_order_sale_week = merge_order_payment_week.groupby('order_purchase_timestamp').agg(
    total_order = ('order_id','count'),
    total_sales = ('payment_value','sum')).reset_index()
result_order_sale_week = result_order_sale_week.rename(columns ={'order_purchase_timestamp':'day_of_week'})
result_order_sale_week

,day_of_week,total_order,total_sales
0,3,103886,16008872.12


In [15]:
# Average Rating Based on the Difference Between the Estimated and Actual Delivery Dates
# Difference Between the Estimated and Actual Delivery Dates
df_orders['Difference_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date']) - pd.to_datetime(df_orders['order_delivered_customer_date'])

"""" Categorize the delivery status based on the following conditions:
If the difference is more than 10 days, categorize it as “late over 10 days”.
If the difference is between 5 and 10 days, categorize it as “late from 5 days to 10 days”.
If the difference is less than 5 days, categorize it as “late under 5 days”.
If the difference is 0 or greater, categorize it as “on time delivery”.
Calculate the average rating based on the est_to_deliver_detail column. """
def delivery_status (time):
  if time > 10:
    return 'late over 10 days'
  elif 5 <=  time <= 10:
    return 'late from 5 days to 10 days'
  elif time < 5:
    return 'late under 5 days'
  else:
    return 'on time delivery'
# Extract daterime component
df_orders['delivery_category'] = df_orders['Difference_date'].dt.days
# Apply delivery status caterogy
df_orders['delivery_category'] = df_orders['delivery_category'].apply(delivery_status)


#Calculate the average rating based on the est_to_deliver_detail
merge_order_review = pd.merge(df_orders,df_order_reviews, how = 'inner', on = 'order_id')
merge_order_review = merge_order_review.groupby('delivery_category').agg(
    avg_rating = ('review_score','mean')).reset_index()
merge_order_review = merge_order_review.round(2)
merge_order_review


,delivery_category,avg_rating
0,late from 5 days to 10 days,4.28
1,late over 10 days,4.32
2,late under 5 days,3.41
3,on time delivery,1.76
